# 🏋️ Fun with Text and Image Embeddings 🍎

Welcome to our **Health & Fitness** embeddings notebook! In this tutorial, we'll show you how to:

1. **Initialize** an `AIProjectClient` to access your Azure AI Foundry project.
2. **Embed text** using the project's OpenAI client (`get_openai_client().embeddings`) with our fun health-themed phrases.
3. **Use a prompt template** for extra context.

Let's get started and have some fun with our healthy ideas! 🍏

> **Disclaimer**: This notebook is for educational purposes only. Always consult a professional for medical advice.

<img src="./seq-diagrams/2-embeddings.png" width="30%"/>

## 1. Setup & Environment

#### Prerequisites:
- Deploy a text embeddings model (**text-embedding-3-large**) in Microsoft Foundry, already performed in Exercise 1, Task 1.

#
We'll import our libraries and load the environment variables for:
- `PROJECT_ENDPOINT`: Your Foundry project endpoint.
- `AZURE_OPENAI_ENDPOINT`: Your Azure OpenAI endpoint (embeddings are served from here).
- `EMBEDDING_MODEL_DEPLOYMENT_NAME`: The text embeddings model deployment name.

We'll import libraries, load environment variables, create an `AIProjectClient`, and build a keyless (Microsoft Entra ID) Azure OpenAI client for the embeddings calls.

> **Note:** The embeddings API is served from the **Azure OpenAI endpoint** rather than the project's `/openai/v1` route, so this notebook builds an `AzureOpenAI` client directly (still keyless — no API key needed). Authentication uses your `az login` identity.

> #### Complete [1-basic-chat-completion.ipynb](./1-basic-chat-completion.ipynb) notebook before starting this one
Let's begin! 🚀

In [ ]:
import os
import re
from dotenv import load_dotenv
from pathlib import Path

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.ai.projects import AIProjectClient
from openai import AzureOpenAI

# Load environment variables
notebook_path = Path().absolute()
parent_dir = notebook_path.parent.parent
load_dotenv(parent_dir / '.env')

# Embedding model deployment name (from Build > Models in the Foundry portal)
EMBEDDING_MODEL_DEPLOYMENT_NAME = os.environ.get("EMBEDDING_MODEL_DEPLOYMENT_NAME", "text-embedding-3-large")

try:
    # The project client confirms your Foundry project is reachable (used across the workshop).
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )

    # Embeddings are served from the Azure OpenAI endpoint. We build a keyless (Microsoft Entra ID)
    aoai_endpoint = re.match(r"https://[^/]+", os.environ["AZURE_OPENAI_ENDPOINT"]).group(0)
    token_provider = get_bearer_token_provider(
        DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
    )
    openai_client = AzureOpenAI(
        azure_endpoint=aoai_endpoint,
        azure_ad_token_provider=token_provider,
        api_version=os.environ.get("OPENAI_API_VERSION", "2024-10-21"),
    )
    print("🎉 Successfully created AIProjectClient + Azure OpenAI client (for embeddings)")
except Exception as e:
    print("❌ Error creating client:", e)

## 2. Text Embeddings

We'll call `client.embeddings.create()` from our `AzureOpenAI` to retrieve the embeddings client. Then we'll embed some fun health-themed phrases:

- "🍎 An apple a day keeps the doctor away"
- "🏋️ 15-minute HIIT workout routine"
- "🧘 Mindful breathing exercises"

The output will be numeric vectors representing each phrase in semantic space. Let’s see those embeddings!

In [ ]:
text_phrases = [
    "An apple a day keeps the doctor away 🍎",
    "Quick 15-minute HIIT workout routine 🏋️",
    "Mindful breathing exercises 🧘"
]

try:
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL_DEPLOYMENT_NAME,
        input=text_phrases
    )

    for item in response.data:
        vec = item.embedding
        sample_str = f"[{vec[0]:.4f}, {vec[1]:.4f}, ..., {vec[-2]:.4f}, {vec[-1]:.4f}]"
        print(
            f"Sentence {item.index}: '{text_phrases[item.index]}':\n"
            f"  Embedding length={len(vec)}\n"
            f"  Sample: {sample_str}\n"
        )

except Exception as e:
    print("❌ Error embedding text:", e)

## 3. Prompt Template Example 📝

Even though our focus is on embeddings, here's how you might prepend some context to a user message. Imagine you want to embed user text but first add a system prompt such as “You are HealthFitGPT, a fitness guidance model…” This little extra helps set the stage for more context-aware embeddings.


In [ ]:
# A basic prompt template (system-style) we'll prepend to user text.
TEMPLATE_SYSTEM = (
    "You are HealthFitGPT, a fitness guidance model.\n"
    "Please focus on healthy advice and disclaim you're not a doctor.\n\n"
    "User message:"  # We'll append the user message after this.
)

def embed_with_template(user_text):
    content = TEMPLATE_SYSTEM + " " + user_text

    rsp = openai_client.embeddings.create(
        model=EMBEDDING_MODEL_DEPLOYMENT_NAME,
        input=[content]
    )

    return rsp.data[0].embedding

sample_user_text = "Can you suggest a quick home workout for busy moms?"
embedding_result = embed_with_template(sample_user_text)
print("Embedding length:", len(embedding_result))
print("First few dims:", embedding_result[:8])

## 4. Wrap-Up & Next Steps
🎉 We've shown how to:
- Set up the `AIProjectClient`.
- Get **text embeddings** using *text-embedding-3-large*.
- Use a **prompt template** to add system context to your embeddings.

**Where to go next?**
- Explore `azure-ai-evaluation` for evaluating your embeddings.
- Use `azure-core-tracing-opentelemetry` for end-to-end telemetry.
- Build out a retrieval pipeline to compare similarity of embeddings.

Have fun experimenting, and remember: when it comes to your health, always consult a professional!